In [1]:
year = 2007
month = 1

In [2]:
# Parameters
year = 1996
month = 3


In [3]:
import numpy as np
import pandas as pd
import xarray as xr
import os, pathlib, stat, textwrap
import calendar
import datetime
from datetime import date

### URLs

In [4]:
# Ufiles = "https://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridU"
Ufiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridU"
Vfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridV"
Wfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridW"
Tfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridT"
Sfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridS"
# #mesh url
# Zgr = "http://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1/glorys12v1-pgnstatics/PSY4V3R1_mesh_zgr.nc"
# Hgr = "http://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1/glorys12v1-pgnstatics/PSY4V3R1_mesh_hgr.nc"

### Environment 

In [5]:
os.environ["NETRC"] = "/home/b/b383184/.netrc"

### Mesh

In [6]:
ds_Zgr = xr.open_dataset('../data/Zgr_mesh.nc')
ds_Zgr

<xarray.Dataset> Size: 555MB
Dimensions:       (y: 3059, x: 4322, z: 50, t: 1)
Dimensions without coordinates: y, x, z, t
Data variables: (12/13)
    nav_lon       (y, x) float32 53MB ...
    nav_lat       (y, x) float32 53MB ...
    nav_lev       (z) float32 200B ...
    time_counter  (t) float64 8B ...
    mbathy        (t, y, x) int16 26MB ...
    hdept         (t, y, x) float64 106MB ...
    ...            ...
    e3t_ps        (t, y, x) float64 106MB ...
    e3w_ps        (t, y, x) float64 106MB ...
    gdept_0       (t, z) float64 400B ...
    gdepw_0       (t, z) float64 400B ...
    e3t_0         (t, z) float64 400B ...
    e3w_0         (t, z) float64 400B ...
Attributes:
    file_name:            mesh_zgr.nc
    TimeStamp:            03/02/2016 10:24:41 -0000
    Unlimited_Dimension:  t

In [7]:
ds_Hgr = xr.open_dataset('../data/Hgr_mesh.nc')
ds_Hgr

<xarray.Dataset> Size: 1GB
Dimensions:       (y: 3059, x: 4322, z: 50, t: 1)
Dimensions without coordinates: y, x, z, t
Data variables: (12/21)
    nav_lon       (y, x) float32 53MB ...
    nav_lat       (y, x) float32 53MB ...
    nav_lev       (z) float32 200B ...
    time_counter  (t) float64 8B ...
    glamt         (t, y, x) float32 53MB ...
    glamu         (t, y, x) float32 53MB ...
    ...            ...
    e1f           (t, y, x) float64 106MB ...
    e2t           (t, y, x) float64 106MB ...
    e2u           (t, y, x) float64 106MB ...
    e2v           (t, y, x) float64 106MB ...
    e2f           (t, y, x) float64 106MB ...
    ff            (t, y, x) float64 106MB ...
Attributes:
    file_name:            mesh_hgr.nc
    TimeStamp:            03/02/2016 10:24:55 -0000
    Unlimited_Dimension:  t

### Functions

In [8]:
lon0, lon1 = -95, 10
lat0, lat1 = -10, 30

# 1) Use T-point lon/lat
lonT = ds_Hgr.glamt.isel(t=0)
latT = ds_Hgr.gphit.isel(t=0)

# 2) Build boolean mask for your box
mask = (lonT >= lon0) & (lonT <= lon1) & (latT >= lat0) & (latT <= lat1)

# 3) Get index ranges
yy, xx = np.where(mask.values)

y0, y1 = int(yy.min()), int(yy.max())
x0, x1 = int(xx.min()), int(xx.max())

x0, x1, y0, y1

(2305, 3565, 1374, 1873)

In [9]:
last_day = calendar.monthrange(year, month)[1]
start_date = datetime.datetime(year, month, 1)
end_date = datetime.datetime(year, month, last_day)

print(end_date.strftime("%Y-%m-%d"))

1996-03-31


In [10]:
def glorys_days(start_date, end_date):
    return pd.date_range(start=start_date, end=end_date, freq="D") + pd.Timedelta(hours=12)

days = glorys_days(start_date.strftime("%Y-%m-%d")
                   , end_date.strftime("%Y-%m-%d"))

In [11]:
def download_MERCATOR(url, varname, starts, ends, x0, x1, y0, y1, output_file):

    from tqdm import tqdm
    import xarray as xr
    
    parts = []
    for tt in tqdm(range(len(days)//2)):
        da = (
            xr.open_dataset(url, engine="pydap", mask_and_scale=False, decode_cf=True)[varname]
            .sortby("time_counter")
            .isel(x=slice(x0, x1), y=slice(y0, y1))
            .sel(time_counter=slice(starts[tt], ends[tt]))
            .astype("float32")
            .load()
        )
        parts.append(da)
    
    da_all = xr.concat(parts, dim="time_counter")
    da_all.to_dataset(name=varname).to_netcdf(output_file, unlimited_dims=["time_counter"])
    print(f"Saved {output_file}")

In [12]:
starts = days[0::2]
ends = days[1::2].tolist()  
ends[-1] = days[-1]
ends

for tt in  range(len(days)//2):
    print('start_date '+str(starts[tt]))
    print('end_date '+str(ends[tt]))

start_date 1996-03-01 12:00:00
end_date 1996-03-02 12:00:00
start_date 1996-03-03 12:00:00
end_date 1996-03-04 12:00:00
start_date 1996-03-05 12:00:00
end_date 1996-03-06 12:00:00
start_date 1996-03-07 12:00:00
end_date 1996-03-08 12:00:00
start_date 1996-03-09 12:00:00
end_date 1996-03-10 12:00:00
start_date 1996-03-11 12:00:00
end_date 1996-03-12 12:00:00
start_date 1996-03-13 12:00:00
end_date 1996-03-14 12:00:00
start_date 1996-03-15 12:00:00
end_date 1996-03-16 12:00:00
start_date 1996-03-17 12:00:00
end_date 1996-03-18 12:00:00
start_date 1996-03-19 12:00:00
end_date 1996-03-20 12:00:00
start_date 1996-03-21 12:00:00
end_date 1996-03-22 12:00:00
start_date 1996-03-23 12:00:00
end_date 1996-03-24 12:00:00
start_date 1996-03-25 12:00:00
end_date 1996-03-26 12:00:00
start_date 1996-03-27 12:00:00
end_date 1996-03-28 12:00:00
start_date 1996-03-29 12:00:00
end_date 1996-03-31 12:00:00


### Data download

In [13]:
U_out = f'U_{start_date.strftime("%Y-%m")}.nc'
V_out = f'V_{start_date.strftime("%Y-%m")}.nc'
W_out = f'W_{start_date.strftime("%Y-%m")}.nc'
T_out = f'T_{start_date.strftime("%Y-%m")}.nc'
S_out = f'S_{start_date.strftime("%Y-%m")}.nc'

outpath = '/work/bk1450/b383184/Amazon/Mercator/data/variables'

In [14]:
#U 
download_MERCATOR(
    Ufiles, "vozocrtx", starts, ends, x0, x1, y0, y1,outpath+U_out
)

  0%|                                                                                                                           | 0/15 [00:00<?, ?it/s]

  7%|███████▋                                                                                                           | 1/15 [00:20<04:42, 20.21s/it]

 13%|███████████████▎                                                                                                   | 2/15 [00:39<04:19, 19.94s/it]

 20%|███████████████████████                                                                                            | 3/15 [00:58<03:50, 19.24s/it]

 27%|██████████████████████████████▋                                                                                    | 4/15 [01:34<04:44, 25.82s/it]

...

 87%|██████████████████████████████████████████████████████████████████████████████████████████████████▊               | 13/15 [05:14<00:49, 24.54s/it]

 93%|██████████████████████████████████████████████████████████████████████████████████████████████████████████▍       | 14/15 [05:48<00:27, 27.33s/it]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15/15 [06:18<00:00, 28.13s/it]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15/15 [06:18<00:00, 25.22s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variablesU_1996-03.nc


In [15]:
download_MERCATOR(
    Vfiles, "vomecrty", starts, ends, x0, x1, y0, y1,outpath+V_out 
)

  0%|                                                                                                                           | 0/15 [00:00<?, ?it/s]

  7%|███████▌                                                                                                          | 1/15 [03:01<42:15, 181.14s/it]

 13%|███████████████▎                                                                                                   | 2/15 [03:22<18:53, 87.19s/it]

 20%|███████████████████████                                                                                            | 3/15 [03:46<11:41, 58.45s/it]

 27%|██████████████████████████████▋                                                                                    | 4/15 [04:18<08:46, 47.89s/it]

...

 87%|██████████████████████████████████████████████████████████████████████████████████████████████████▊               | 13/15 [10:29<01:12, 36.46s/it]

 93%|██████████████████████████████████████████████████████████████████████████████████████████████████████████▍       | 14/15 [10:55<00:33, 33.43s/it]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15/15 [11:27<00:00, 32.81s/it]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15/15 [11:27<00:00, 45.82s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variablesV_1996-03.nc


In [16]:
download_MERCATOR(
    Wfiles, "vovecrtz", starts, ends, x0, x1, y0, y1,outpath+W_out 
)

  0%|                                                                                                                           | 0/15 [00:00<?, ?it/s]

  7%|███████▋                                                                                                           | 1/15 [00:26<06:17, 26.96s/it]

 13%|███████████████▎                                                                                                   | 2/15 [01:00<06:40, 30.83s/it]

 20%|███████████████████████                                                                                            | 3/15 [03:42<18:10, 90.87s/it]

 27%|██████████████████████████████▋                                                                                    | 4/15 [04:03<11:34, 63.13s/it]

...

 87%|██████████████████████████████████████████████████████████████████████████████████████████████████▊               | 13/15 [10:07<00:57, 28.70s/it]

 93%|██████████████████████████████████████████████████████████████████████████████████████████████████████████▍       | 14/15 [10:28<00:26, 26.24s/it]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15/15 [11:08<00:00, 30.59s/it]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15/15 [11:08<00:00, 44.60s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variablesW_1996-03.nc


In [17]:
download_MERCATOR(
    Tfiles, "votemper", starts, ends, x0, x1, y0, y1,outpath+T_out
)

  0%|                                                                                                                           | 0/15 [00:00<?, ?it/s]

  7%|███████▋                                                                                                           | 1/15 [01:19<18:33, 79.52s/it]

 13%|███████████████▎                                                                                                   | 2/15 [02:47<18:21, 84.73s/it]

 20%|███████████████████████                                                                                            | 3/15 [03:23<12:25, 62.11s/it]

 27%|██████████████████████████████▋                                                                                    | 4/15 [03:43<08:24, 45.83s/it]

...

 87%|██████████████████████████████████████████████████████████████████████████████████████████████████▊               | 13/15 [07:29<00:52, 26.08s/it]

 93%|██████████████████████████████████████████████████████████████████████████████████████████████████████████▍       | 14/15 [08:01<00:27, 27.86s/it]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15/15 [08:33<00:00, 29.24s/it]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15/15 [08:33<00:00, 34.26s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variablesT_1996-03.nc


In [18]:
download_MERCATOR(
    Sfiles, "vosaline", starts, ends, x0, x1, y0, y1,outpath+S_out 
)

  0%|                                                                                                                           | 0/15 [00:00<?, ?it/s]

  7%|███████▋                                                                                                           | 1/15 [00:25<05:54, 25.35s/it]

 13%|███████████████▎                                                                                                   | 2/15 [01:13<08:27, 39.05s/it]

 20%|███████████████████████                                                                                            | 3/15 [01:36<06:17, 31.45s/it]

 27%|██████████████████████████████▋                                                                                    | 4/15 [01:55<04:53, 26.71s/it]

...

 87%|██████████████████████████████████████████████████████████████████████████████████████████████████▊               | 13/15 [05:14<00:43, 21.51s/it]

 93%|██████████████████████████████████████████████████████████████████████████████████████████████████████████▍       | 14/15 [05:36<00:21, 21.65s/it]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15/15 [06:19<00:00, 27.97s/it]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15/15 [06:19<00:00, 25.30s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variablesS_1996-03.nc
